# 🔑 Solutions: Lesson 04 · Gradient Descent & Backpropagation

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 4.1: Gradient descent in 1-D  &nbsp; `⭐`

Write `gd(grad, w0, lr, steps)` that runs gradient descent and returns the final `w`. Test: minimise $f(w)=(w-3)^2+1$ starting from 8.


In [2]:
def gd(grad, w0, lr, steps):
    w = w0
    for _ in range(steps):
        w = w - lr * grad(w)
    return w

In [3]:
# ▶ self-check
res = gd(lambda w: 2 * (w - 3), 8.0, 0.3, 50)
dl.check("converges to w = 3", res is not None and abs(res - 3) < 1e-4)
res2 = gd(lambda w: 4 * w ** 3 - 6 * w, 2.0, 0.01, 500)
dl.check("also works on a non-quadratic: f=w⁴−3w² → w≈√1.5", res2 is not None and abs(abs(res2) - np.sqrt(1.5)) < 1e-3)

✅ converges to w = 3
✅ also works on a non-quadratic: f=w⁴−3w² → w≈√1.5


---
### 🧩 Problem 4.2: Find the critical learning rate  &nbsp; `⭐⭐`

For $f(w)=w^2$ (gradient $2w$), gradient descent updates $w\leftarrow w(1-2\eta)$. By *experimenting* with the provided `final_abs_w(lr)`
function, find the learning rate above which it diverges. Set `lr_threshold` (to within ±0.05).
Then explain to yourself *why* in one sentence (the multiplier is $|1-2\eta|$).


In [4]:
def final_abs_w(lr, steps=60, w0=1.0):
    w = w0
    for _ in range(steps):
        w = w - lr * 2 * w
    return abs(w)

lr_threshold = 1.0

In [5]:
# ▶ self-check
dl.check("lr_threshold ≈ 1.0", lr_threshold is not None and abs(lr_threshold - 1.0) <= 0.05)

✅ lr_threshold ≈ 1.0


**Why this works.** The update multiplies `w` by (1−2η) each step. For η<1 that factor has magnitude <1 (shrinks to 0); for η>1 it exceeds 1 in magnitude (blows up); η=1 flips sign forever.

---
### 🧩 Problem 4.3: Numerical gradient  &nbsp; `⭐⭐`

Write `numerical_gradient(f, x, eps=1e-6)` that returns the gradient of a scalar function `f` at the vector `x` using **central differences**, one coordinate at a time.


In [6]:
def numerical_gradient(f, x, eps=1e-6):
    g = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        xp, xm = x.astype(float).copy(), x.astype(float).copy()
        xp[i] += eps; xm[i] -= eps
        g[i] = (f(xp) - f(xm)) / (2 * eps)
    return g

In [7]:
# ▶ self-check
f = lambda v: v[0] ** 2 + 3 * v[0] * v[1] + np.sin(v[2])
x0 = np.array([1.0, 2.0, 0.5])
g = numerical_gradient(f, x0)
dl.check("matches analytic gradient [2x+3y, 3x, cos z]", g is not None and np.allclose(g, [2 + 6, 3, np.cos(0.5)], atol=1e-5))

✅ matches analytic gradient [2x+3y, 3x, cos z]


---
### 🧩 Problem 4.4: Backward pass of a dense ReLU layer  &nbsp; `⭐⭐⭐`

A dense layer computes `Z = X @ W + b`, `A = relu(Z)`. Given the upstream gradient `dA` (shape of `A`), implement
`dense_relu_backward(dA, X, Z, W)` returning **`(dX, dW, db)`**.
(Use the table in Section 5: first gate by ReLU's slope to get `dZ`, then `dW = Xᵀ dZ`, `db = Σ dZ`, `dX = dZ Wᵀ`.)
The self-check validates your gradients against numerical differentiation.


In [8]:
def dense_relu_backward(dA, X, Z, W):
    dZ = dA * (Z > 0)
    return dZ @ W.T, X.T @ dZ, dZ.sum(axis=0)

In [9]:
# ▶ self-check
rng = np.random.default_rng(0)
X, W, b, R = rng.normal(size=(6, 4)), rng.normal(size=(4, 3)), rng.normal(size=3), rng.normal(size=(6, 3))
Z = X @ W + b
out = dense_relu_backward(R, X, Z, W)       # loss = sum(R * relu(Z))  →  dA = R
ok = out is not None and len(out) == 3
dl.check("returns three arrays with the right shapes", ok and out[0].shape == X.shape and out[1].shape == W.shape and out[2].shape == b.shape)
if ok:
    loss = lambda W_: np.sum(R * np.maximum(0, X @ W_ + b))
    num = np.zeros_like(W); eps = 1e-6
    for i in np.ndindex(*W.shape):
        Wp, Wm = W.copy(), W.copy(); Wp[i] += eps; Wm[i] -= eps
        num[i] = (loss(Wp) - loss(Wm)) / (2 * eps)
    dl.check("dW matches numerical gradient", np.allclose(out[1], num, atol=1e-5))
    dl.check("db matches", np.allclose(out[2], (R * (Z > 0)).sum(0)))
    dl.check("dX matches", np.allclose(out[0], (R * (Z > 0)) @ W.T))

✅ returns three arrays with the right shapes
✅ dW matches numerical gradient
✅ db matches
✅ dX matches


**Why this works.** The same three lines handle *every* dense layer in *every* network: gate by the activation slope, then outer-product-like sums for the weights, and a transposed multiply to send the error to the layer below.

---
### 🧩 Problem 4.5: Teach the autograd engine new tricks  &nbsp; `⭐⭐⭐`

The starter contains a minimal `Value` class (add & multiply already work). **Implement** `exp()`, `__neg__` and `__sub__`
so that `backward()` produces correct gradients. Recall: $\frac{d}{dx}e^x=e^x$.
Test function: $f(x)=(e^x-1)\cdot(-x)$ at $x=0.5$.


In [10]:
class Value:
    def __init__(self, data, _children=()):
        self.data, self.grad = float(data), 0.0
        self._prev, self._backward = set(_children), (lambda: None)
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad; other.grad += out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad; other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self); self.grad = 1.0
        for v in reversed(topo): v._backward()

    def exp(self):
        out = Value(np.exp(self.data), (self,))
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    def __neg__(self):
        return self * -1
    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self + (-other)

In [11]:
# ▶ self-check
try:
    x = Value(0.5)
    f = (x.exp() - 1) * (-x)
    f.backward()
    true_grad = -(np.exp(0.5) - 1) - 0.5 * np.exp(0.5)       # d/dx[-(e^x−1)x]
    dl.check("f(0.5) value is correct", np.isclose(f.data, -(np.exp(0.5) - 1) * 0.5))
    dl.check("df/dx is correct", np.isclose(x.grad, true_grad))
except Exception as e:
    dl.check("implement exp, __neg__, __sub__", False, f"{type(e).__name__}: {e}")

✅ f(0.5) value is correct
✅ df/dx is correct


**Why this works.** `exp` is its own derivative, so its local gradient is just its output. Subtraction and negation are *compositions* of operations we already have: autograd gets their derivatives for free. That compositionality is the whole point.